In [1]:
# this is test

In [8]:
import urllib.request
import json
from packaging.specifiers import SpecifierSet
from packaging.version import Version

def analyze_requirements(input_file):
    # Read non-empty lines from the input file
    with open(input_file, 'r') as f:
        lines = [line.strip() for line in f if line.strip() and not line.startswith('#')]

    dependencies_checked = len(lines)
    not_found_on_pypi = 0
    incompatible_python = 0
    
    highest_min_version = None
    highest_req_str = ""

    for line in lines:
        # Parse package name and version (e.g., django==4.2)
        if "==" in line:
            pkg_name, pinned_version = line.split("==")
            pkg_name = pkg_name.strip()
            pinned_version = pinned_version.strip()
        else:
            pkg_name = line.strip()
            pinned_version = None

        # 1. Check if package exists on PyPI
        url = f"https://pypi.org/pypi/{pkg_name}/json"
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        try:
            with urllib.request.urlopen(req) as response:
                data = json.loads(response.read().decode())
        except urllib.error.HTTPError as e:
            if e.code == 404:
                not_found_on_pypi += 1
            continue
        except Exception:
            continue

        # 2. If a pinned version is specified, check if that version exists
        if pinned_version:
            if pinned_version not in data.get("releases", {}):
                not_found_on_pypi += 1
                continue

            # Fetch version-specific metadata
            version_url = f"https://pypi.org/pypi/{pkg_name}/{pinned_version}/json"
            v_req = urllib.request.Request(version_url, headers={'User-Agent': 'Mozilla/5.0'})
            try:
                with urllib.request.urlopen(v_req) as v_resp:
                    v_data = json.loads(v_resp.read().decode())
                    requires_python = v_data.get("info", {}).get("requires_python")
            except Exception:
                requires_python = None

            # 3. Check compatibility with Python 3.11
            if requires_python:
                try:
                    specifier = SpecifierSet(requires_python)
                    # Check if Python 3.11 matches the requirement specifier
                    if not specifier.contains(Version("3.11")):
                        incompatible_python += 1
                    
                    # 4. Find highest requirement
                    for spec in specifier:
                        if spec.operator in ('>=', '>'):
                            min_ver = Version(spec.version)
                            if highest_min_version is None or min_ver > highest_min_version:
                                highest_min_version = min_ver
                                highest_req_str = f"{pkg_name}=={pinned_version} (>={spec.version})"
                except Exception:
                    pass

    # Print results matching the output template
    print(f"Dependencies checked: {dependencies_checked}")
    print(f"Not found on PyPI: {not_found_on_pypi}")
    print(f"Incompatible with Python 3.11: {incompatible_python}")
    print(f"Highest requirement: {highest_req_str}")

if __name__ == "__main__":
    analyze_requirements("requirements.txt")

Dependencies checked: 8
Not found on PyPI: 3
Incompatible with Python 3.11: 1
Highest requirement: scikit-learn==1.4.0 (>=3.9)
